# Tinker from zero to hero

**A complete, from-zero to-hero tutorial for the Kaggle community**

Hey everyone! I'm sharing my full pipeline — dataset prep, cloud LoRA training with [Tinker](https://tinker.thinkingmachines.ai), live evaluation on Tinker's sampling API, and local weight download — so you can reproduce (and hopefully beat!) the **47/50** score.

**Why Tinker?** Because you can fine-tune big models with LoRA **without owning a single GPU**. Tinker runs everything on their cloud — you just send data and hyperparameters through their Python SDK. It's like having an H100 cluster in your pocket (well, in your API key) and also because we got some free credits lol.

### What you'll learn
1. **Getting your Tinker API key** (takes 2 minutes)
2. **Preparing your dataset** in the right SFT format
3. **Training a LoRA adapter** i used in the past on Nemotron-3-Nano-30B via Tinker (full code)
4. **Evaluating your model** on Tinker's sampling API (full code)
5. **Downloading weights locally** for Kaggle submission
6. **Tips & tricks** from my journey

Let's go! i hope you like it fam :)

## Step 0: Install Dependencies

Tinker's SDK is lightweight — it doesn't even need PyTorch installed locally. We also grab `transformers` for the tokenizer and `safetensors` to inspect our downloaded weights later.

> **Kaggle users:** These packages are already available on Kaggle kernels with internet enabled. Just run the cell below.

In [1]:
!pip install -q tinker transformers safetensors requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.0/187.0 kB 4.2 MB/s eta 0:00:00


## Step 1: Get Your Tinker API Key (2 minutes, seriously)

1. Go to **[tinker.thinkingmachines.ai](https://tinker.thinkingmachines.ai)** and sign up (GitHub or Google login works)
2. Once logged in, click your profile icon (top-right) → **"API Keys"**
3. Click **"Create New Key"** — copy the key that starts with `tml-...`
4. Paste it below where it says `YOUR_API_KEY_HERE`

That's it.

> **Security tip:** Never commit your API key to a public repo. Use environment variables or a `.txt` file you `.gitignore`.

In [2]:
import os
import json
import time
import random
import logging
import requests

import tinker
from transformers import AutoTokenizer

# ============================================================
#  PUT YOUR TINKER API KEY HERE
# ============================================================
os.environ["TINKER_API_KEY"] = "YOUR_API_KEY_HERE"   # <-- replace this!

# Model we're fine-tuning
MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("tinker-tutorial")

print("Tinker SDK version:", tinker.__version__)
print("API key set:", "TINKER_API_KEY" in os.environ and os.environ["TINKER_API_KEY"] != "YOUR_API_KEY_HERE")

Tinker SDK version: 0.16.1
API key set: False


## Step 2: Prepare Your Dataset

The key to a good score is a **high-quality SFT dataset** where every row follows this exact format:

```
<|im_start|>user
{math problem}
<|im_end|>
<|im_start|>assistant
<think>
{step-by-step reasoning}
</think>
\boxed{final_answer}
<|im_end|>
```

**Why this format?**
- `<|im_start|>` / `<|im_end|>` are the chat template tokens Nemotron expects
- `<think>...</think>` teaches the model to reason before answering (chain-of-thought)
- `\boxed{...}` is the standard math answer format the competition evaluator looks for

I have built a 10,000-row dataset from multiple high-quality sources:
- **s1K-1.1** (simplescaling) — 1,000 curated hard math problems
- **LIMO** (GAIR) — 817 competition-grade problems
- **OpenThoughts-114k** — diverse reasoning examples well selected
- **OpenMathReasoning** (NVIDIA) — official NVIDIA math reasoning data

Below I show how to load and validate a JSONL conversation file (the format Tinker expects).

In [3]:
# ============================================================
#  Dataset: load from JSONL (one JSON object per line)
#  Each object = {"messages": [{"role": "user", "content": ...}, {"role": "assistant", "content": ...}]}
# ============================================================

def load_conversations(path):
    """Load JSONL conversation file."""
    conversations = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                conversations.append(json.loads(line))
    return conversations


# If you already have a JSONL file, load it directly:
# all_convos = load_conversations("sft_10k_final.jsonl")

# --- OR build from CSV (our format: single 'text' column with full chat template) ---
def csv_to_conversations(csv_path):
    """Convert a CSV with a 'text' column (containing the full chat-template string) into conversations."""
    import csv
    convos = []
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            text = row["text"]
            messages = []
            # Parse im_start blocks
            parts = text.split("<|im_start|>")
            for part in parts:
                part = part.strip()
                if not part:
                    continue
                # Remove trailing <|im_end|>
                part = part.replace("<|im_end|>", "").strip()
                if part.startswith("user"):
                    content = part[len("user"):].strip()
                    messages.append({"role": "user", "content": content})
                elif part.startswith("assistant"):
                    content = part[len("assistant"):].strip()
                    messages.append({"role": "assistant", "content": content})
                elif part.startswith("system"):
                    content = part[len("system"):].strip()
                    messages.append({"role": "system", "content": content})
            if messages:
                convos.append({"messages": messages})
    return convos

# Example: loading from our CSV
# all_convos = csv_to_conversations("sft_10k_final.csv")

print("Dataset loading functions ready!")
print("Tip: Use load_conversations() for JSONL, csv_to_conversations() for CSV")

Dataset loading functions ready!
Tip: Use load_conversations() for JSONL, csv_to_conversations() for CSV


## Step 3: Build Training Datums (The Secret Sauce)

Here's where most people get tripped up. Tinker needs a special `tinker.Datum` object for each training example. The default renderer from `tinker_cookbook` actually **strips the `<think>` reasoning** — which kills chain-of-thought training!

My fix: **build datums manually** so the full reasoning is preserved and weighted for training.

**How it works:**
- I tokenize the full conversation (user + assistant with `<think>` + `\boxed{}`)
- User/system tokens get weight `0` (don't train on the question)
- Assistant tokens get weight `1` (train on the reasoning + answer)
- I also build `target_tokens` (input shifted by 1) for next-token prediction

In [4]:
# ============================================================
#  Tokenizer + Manual Datum Builder
# ============================================================

print("Loading tokenizer (this downloads ~500MB the first time)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
print(f"Tokenizer loaded! Vocab size: {tokenizer.vocab_size:,}")


def build_datum_manual(tokenizer, messages, max_length=8192):
    """Build a tinker.Datum manually, preserving full <think> reasoning.
    Trains on assistant tokens only (user/system get weight=0).
    
    The Tinker cross_entropy loss requires both:
      - weights (float32): per-token loss weights
      - target_tokens (int64): input tokens shifted by 1 (next-token targets)
    """
    # Render full conversation using the tokenizer's chat template
    full_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    all_tokens = tokenizer.encode(full_text)

    # Truncate if needed
    if len(all_tokens) > max_length:
        all_tokens = all_tokens[:max_length]

    # Find where the assistant content starts by tokenizing user-only part
    user_only_msgs = [m for m in messages if m["role"] != "assistant"]
    user_text = tokenizer.apply_chat_template(
        user_only_msgs, tokenize=False, add_generation_prompt=True
    )
    user_tokens = tokenizer.encode(user_text)
    n_user = min(len(user_tokens), len(all_tokens))

    # Weights: 0 for user/system, 1 for assistant (shifted by 1 for next-token prediction)
    weights = [0.0] * n_user + [1.0] * (len(all_tokens) - n_user)
    weights = weights[1:] + [0.0]  # shift: predict next token

    # Target tokens: input shifted by 1 (next-token prediction targets)
    target_tokens = list(all_tokens[1:]) + [0]

    chunk = tinker.EncodedTextChunk(tokens=all_tokens)
    model_input = tinker.ModelInput(chunks=[chunk])
    tensor_weights = tinker.TensorData(
        data=weights, shape=[len(weights)], dtype="float32"
    )
    tensor_targets = tinker.TensorData(
        data=target_tokens, shape=[len(target_tokens)], dtype="int64"
    )
    return tinker.Datum(
        model_input=model_input,
        loss_fn_inputs={"weights": tensor_weights, "target_tokens": tensor_targets},
    )


# Quick test with a dummy example
test_msgs = [
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "<think>\n2+2 = 4\n</think>\n\\boxed{4}"}
]
test_datum = build_datum_manual(tokenizer, test_msgs)
print(f"\nTest datum created! Token count: {test_datum.model_input.length}")
print("build_datum_manual() is working correctly!")

Loading tokenizer (this downloads ~500MB the first time)...


config.json: 0.00B [00:00, ?B/s]

configuration_nemotron_h.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16:
- configuration_nemotron_h.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

2026-03-24 19:05:31,134 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/420 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Tokenizer loaded! Vocab size: 131,072

Test datum created! Token count: 40
build_datum_manual() is working correctly!


## Step 4: Train Your LoRA on Tinker (The Fun Part)

This is where the magic happens. We're going to:

1. **Connect** to Tinker's API
2. **Create a LoRA training client** (rank=32)
3. **Run a full training loop** with linear warmup + decay, periodic evaluation, and checkpoint saving

The whole thing runs on Tinker's GPUs — your laptop just sends HTTP requests and you drink coffee (or tea :).

### Hyperparameters I used (feel free to experiment!):

| Parameter | Value | Why |
|-----------|-------|-----|
| LoRA rank | 32 | my favorite |
| Batch size | 64 | Larger = more stable gradients |
| Learning rate | 2e-4 | Sweet spot for LoRA SFT |
| Epochs | 2 | Enough for 10k rows without overfitting |
| Warmup | 5% of steps | Prevents early instability |
| LR schedule | Linear decay | Smooth convergence |
| Adam betas | (0.9, 0.95) | Slightly less momentum on second moment |
| Max seq length | 8192 | Long enough for complex reasoning chains |

In [5]:
# ============================================================
#  FULL TRAINING CODE — This is what I actually ran
# ============================================================

def compute_nll(fwd_bwd_result, batch):
    """Compute mean negative log-likelihood from forward_backward output."""
    total_nll = 0.0
    total_weight = 0.0
    for output, datum in zip(fwd_bwd_result.loss_fn_outputs, batch):
        logprobs = output["logprobs"].data
        w = datum.loss_fn_inputs["weights"].data
        for lp, wi in zip(logprobs, w):
            if wi > 0:
                total_nll -= lp
                total_weight += wi
    return total_nll / max(total_weight, 1.0)


def train_lora(all_convos, tokenizer, epochs=2, lr=2e-4, batch_size=64,
               lora_rank=32, max_length=8192, save_every=10, eval_split=5):
    """
    Full LoRA SFT training loop on Tinker.
    
    Args:
        all_convos: list of {"messages": [...]} dicts
        tokenizer: HuggingFace tokenizer
        epochs: number of training epochs
        lr: peak learning rate
        batch_size: examples per step
        lora_rank: LoRA rank
        max_length: max sequence length in tokens
        save_every: save checkpoint every N steps
        eval_split: number of examples to hold out for eval
    """
    # ---- Convert to Datums ----
    print(f"Converting {len(all_convos)} conversations to training datums...")
    all_datums = []
    skipped = 0
    for i, convo in enumerate(all_convos):
        try:
            datum = build_datum_manual(tokenizer, convo["messages"], max_length)
            all_datums.append(datum)
        except Exception as e:
            skipped += 1
            if skipped <= 3:
                print(f"  Skipped conversation {i}: {e}")

    print(f"  Converted {len(all_datums)} datums ({skipped} skipped)")

    # Token stats
    token_counts = [d.model_input.length for d in all_datums]
    print(f"  Token stats: mean={sum(token_counts)/len(token_counts):.0f}, "
          f"max={max(token_counts)}, min={min(token_counts)}")

    # ---- Train/eval split ----
    random.seed(42)
    indices = list(range(len(all_datums)))
    random.shuffle(indices)
    eval_size = min(eval_split, len(all_datums) // 10)
    eval_indices = indices[:eval_size]
    train_indices = indices[eval_size:]
    eval_datums = [all_datums[i] for i in eval_indices]
    train_datums = [all_datums[i] for i in train_indices]

    n_batches_per_epoch = len(train_datums) // batch_size
    total_steps = n_batches_per_epoch * epochs
    print(f"  Train: {len(train_datums)} | Eval: {len(eval_datums)} | "
          f"Steps/epoch: {n_batches_per_epoch} | Total steps: {total_steps}")

    # ---- Connect to Tinker ----
    print("\nConnecting to Tinker API...")
    service_client = tinker.ServiceClient()
    training_client = service_client.create_lora_training_client(
        base_model=MODEL_NAME,
        rank=lora_rank,
    )
    print("  Connected! Training client created.")

    # ---- Training loop ----
    print(f"\n{'='*60}")
    print(f"  STARTING TRAINING: {total_steps} steps over {epochs} epochs")
    print(f"{'='*60}\n")

    global_step = 0
    best_eval_nll = float("inf")
    train_losses = []
    t_start = time.time()

    for epoch in range(epochs):
        epoch_indices = list(range(len(train_datums)))
        random.seed(epoch)
        random.shuffle(epoch_indices)

        for batch_idx in range(n_batches_per_epoch):
            step_start = time.time()

            # Learning rate schedule: linear warmup (5%) + linear decay
            warmup_steps = max(1, int(0.05 * total_steps))
            if global_step < warmup_steps:
                lr_mult = global_step / warmup_steps
            else:
                lr_mult = max(0.0, 1.0 - (global_step - warmup_steps) / (total_steps - warmup_steps))
            current_lr = lr * lr_mult

            adam_params = tinker.AdamParams(
                learning_rate=current_lr,
                beta1=0.9,
                beta2=0.95,
                eps=1e-8,
            )

            # Get batch
            start_idx = batch_idx * batch_size
            end_idx = start_idx + batch_size
            batch_row_indices = epoch_indices[start_idx:end_idx]
            batch = [train_datums[i] for i in batch_row_indices]

            # Forward + backward + optimizer step
            fwd_bwd_future = training_client.forward_backward(batch, loss_fn="cross_entropy")
            optim_future = training_client.optim_step(adam_params)
            fwd_bwd_result = fwd_bwd_future.result()
            optim_result = optim_future.result()

            # Compute train NLL
            train_nll = compute_nll(fwd_bwd_result, batch)
            train_losses.append(train_nll)
            step_time = time.time() - step_start

            # Log every 5 steps
            if global_step % 5 == 0 or global_step == total_steps - 1:
                elapsed = time.time() - t_start
                avg_loss = sum(train_losses[-10:]) / len(train_losses[-10:])
                print(
                    f"[Step {global_step:4d}/{total_steps}] "
                    f"epoch={epoch+1}/{epochs} "
                    f"lr={current_lr:.2e} "
                    f"train_nll={train_nll:.4f} "
                    f"avg_nll(10)={avg_loss:.4f} "
                    f"step={step_time:.1f}s "
                    f"elapsed={elapsed:.0f}s"
                )

            # Evaluate periodically
            if eval_datums and (global_step % save_every == 0 or global_step == total_steps - 1):
                eval_future = training_client.forward_backward(eval_datums, loss_fn="cross_entropy")
                eval_result = eval_future.result()
                eval_nll = compute_nll(eval_result, eval_datums)
                is_best = eval_nll < best_eval_nll
                if is_best:
                    best_eval_nll = eval_nll
                print(f"  >>> EVAL nll={eval_nll:.4f} {'NEW BEST!' if is_best else ''} (best={best_eval_nll:.4f})")
                # Zero out gradient from eval pass
                training_client.optim_step(
                    tinker.AdamParams(learning_rate=0.0, beta1=0.9, beta2=0.95, eps=1e-8)
                ).result()

            # Save checkpoint periodically
            if save_every > 0 and global_step % save_every == 0 and global_step > 0:
                name = f"step_{global_step:04d}"
                training_client.save_state(name=name).result()
                print(f"  Saved checkpoint: {name}")

            global_step += 1

    # ---- Save final checkpoint ----
    print("\nSaving final checkpoint...")
    training_client.save_state(name="final").result()
    print("  Final state saved!")

    # ---- Save sampler weights (needed for inference + download) ----
    print("Saving sampler weights...")
    sampler_result = training_client.save_weights_for_sampler(name="nemotron_sft_final").result()
    sampler_path = sampler_result.path
    print(f"  Sampler weights saved: {sampler_path}")

    total_time = time.time() - t_start
    print(f"\n{'='*60}")
    print(f"  TRAINING COMPLETE!")
    print(f"{'='*60}")
    print(f"  Total time: {total_time:.0f}s ({total_time/60:.1f} min)")
    print(f"  Total steps: {total_steps}")
    print(f"  Final train NLL: {train_losses[-1]:.4f}")
    print(f"  Best eval NLL: {best_eval_nll:.4f}")
    print(f"  Sampler path: {sampler_path}")

    return service_client, training_client, sampler_path, train_losses


# ============================================================
#  TO ACTUALLY TRAIN, UNCOMMENT THESE LINES:
# ============================================================
# all_convos = load_conversations("sft_10k_final.jsonl")   # or csv_to_conversations(...)
# service_client, training_client, sampler_path, losses = train_lora(
#     all_convos, tokenizer,
#     epochs=2, lr=2e-4, batch_size=64, lora_rank=32, max_length=8192,
# )

print("Training function defined! Uncomment the lines above to run training.")
print("Estimated training time for 10k rows, 2 epochs: ~30-60 minutes on Tinker")

Training function defined! Uncomment the lines above to run training.
Estimated training time for 10k rows, 2 epochs: ~30-60 minutes on Tinker


## Step 5: Test Your Model on Tinker's Sampling API

After training, you can immediately test your model **without downloading anything**. Tinker keeps your LoRA weights on their servers and lets you run inference through their sampling API.

This is how I evaluated my model on 50 hard math problems.

There are two ways to get a sampling client:
1. **Right after training:** `training_client.save_weights_and_get_sampling_client()`
2. **From a saved checkpoint:** `service_client.create_sampling_client(model_path=sampler_path)`

Below is my full evaluation code — it handles answer extraction, `\boxed{}` parsing, and even a recovery mechanism when the model forgets to output a clean answer.

In [6]:
# ============================================================
#  FULL EVALUATION CODE — Test your fine-tuned model on Tinker
# ============================================================
import re
import csv


def extract_boxed_balanced(text):
    """Extract content from the last \\boxed{...} in text, handling nested braces."""
    key = r"\boxed{"
    idx = text.rfind(key)
    if idx < 0:
        return None
    i = idx + len(key)
    depth = 1
    start = i
    while i < len(text) and depth:
        c = text[i]
        if c == "{":
            depth += 1
        elif c == "}":
            depth -= 1
        i += 1
    if depth != 0:
        return None
    return text[start : i - 1].strip()


def extract_predicted_answer(text):
    """Extract answer: tries <answer>...</answer> first, then \\boxed{...}."""
    # Try <answer> tags (last valid one wins)
    last_good = None
    for m in re.finditer(r"<answer>\s*([\s\S]*?)\s*</answer>", text, re.IGNORECASE):
        candidate = m.group(1).strip()
        if candidate and len(candidate) < 800:
            last_good = candidate
    if last_good:
        return last_good
    # Fall back to \boxed{}
    b = extract_boxed_balanced(text)
    if b and len(b) < 800:
        return b
    return None


def normalize_compare(pred, gold):
    """Compare predicted and gold answers with tolerance."""
    p, g = pred.strip(), gold.strip()
    if p == g:
        return True
    if p.replace(" ", "") == g.replace(" ", ""):
        return True
    try:
        pf, gf = float(p), float(g)
        if abs(pf - gf) <= 1e-9 * max(1.0, abs(gf)):
            return True
    except ValueError:
        pass
    return p.lower() == g.lower()


def evaluate_on_tinker(sampling_client, tokenizer, csv_path, 
                       max_new_tokens=6144, temperature=0.2, top_p=0.95, seed=42, limit=0):
    """
    Evaluate the fine-tuned model on a CSV of math problems.
    CSV must have columns: id, problem, answer
    
    Returns: (correct_count, total_count, results_list)
    """
    INSTRUCTION = (
        "Solve the problem. Put your reasoning inside <think>...</think> tags. "
        "Immediately after </think>, output exactly one line: <answer>X</answer> "
        "where X is ONLY your final answer."
    )
    
    rows = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
    if limit:
        rows = rows[:limit]
    
    correct = 0
    results = []
    t0 = time.time()
    
    for i, row in enumerate(rows):
        pid = row["id"]
        problem = row["problem"]
        gold = str(row["answer"]).strip()
        
        # Build prompt
        user_content = f"{INSTRUCTION}\n\n{problem}"
        messages = [{"role": "user", "content": user_content}]
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompt_ids = tokenizer.encode(prompt_text)
        
        # Sample from model
        inp = tinker.ModelInput(chunks=[tinker.EncodedTextChunk(tokens=prompt_ids)])
        params = tinker.SamplingParams(
            max_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            seed=seed + i,
        )
        out = sampling_client.sample(prompt=inp, num_samples=1, sampling_params=params).result()
        gen_ids = out.sequences[0].tokens
        raw = tokenizer.decode(gen_ids)
        
        # Extract and compare answer
        pred = extract_predicted_answer(raw)
        ok = pred is not None and normalize_compare(pred, gold)
        if ok:
            correct += 1
        
        results.append({"id": pid, "gold": gold, "predicted": pred, "correct": ok})
        
        print(f"[{i+1}/{len(rows)}] {pid} | correct={ok} | pred={repr(pred)[:60]} | gold={repr(gold)[:40]}")
    
    elapsed = time.time() - t0
    print(f"\n{'='*60}")
    print(f"  SCORE: {correct}/{len(rows)} ({100*correct/len(rows):.1f}%)")
    print(f"  Time: {elapsed:.1f}s ({elapsed/len(rows):.1f}s per problem)")
    print(f"{'='*60}")
    
    return correct, len(rows), results


# ============================================================
#  TO RUN EVALUATION, UNCOMMENT THESE LINES:
# ============================================================
# Option A: Right after training (sampling_client from training)
# sampling_client = training_client.save_weights_and_get_sampling_client(name="eval_test")
# correct, total, results = evaluate_on_tinker(
#     sampling_client, tokenizer, "hard_50_math_problems_set_v6 (1).csv"
# )

# Option B: From a saved sampler checkpoint path
# service_client = tinker.ServiceClient()
# sampling_client = service_client.create_sampling_client(model_path=sampler_path, base_model=None)
# correct, total, results = evaluate_on_tinker(
#     sampling_client, tokenizer, "hard_50_math_problems_set_v6 (1).csv"
# )

print("Evaluation function defined!")
print("Uncomment the lines above (Option A or B) to run evaluation after training.")

Evaluation function defined!
Uncomment the lines above (Option A or B) to run evaluation after training.


## Step 6: Download Your Weights Locally

Once training is done, you'll want to download the LoRA weights so you can:
- Submit them to the Kaggle competition
- Run local inference
- Share them with the community

Tinker stores checkpoints as `.tar` archives containing:
- `adapter_config.json` — LoRA configuration (rank, alpha, target modules)
- `adapter_model.safetensors` — the actual trained weights (~1.5 GB for rank 32)

### Two download methods:

**Method 1: From the sampler path** (if you just finished training)

**Method 2: Auto-discover checkpoints** (if you ran training earlier and want to find your checkpoints)

In [7]:
# ============================================================
#  DOWNLOAD WEIGHTS — Method 1: From sampler path (after training)
# ============================================================

def download_checkpoint(service_client, sampler_path, output_file="nemotron_sft_lora.tar"):
    """Download a Tinker checkpoint archive to a local .tar file."""
    print(f"Getting download URL for: {sampler_path}")
    rest_client = service_client.create_rest_client()
    url_resp = rest_client.get_checkpoint_archive_url_from_tinker_path(sampler_path).result()
    
    print("Downloading...")
    r = requests.get(url_resp.url, stream=True)
    r.raise_for_status()
    total_bytes = int(r.headers.get("content-length", 0))
    print(f"  File size: {total_bytes / 1e9:.2f} GB")
    
    downloaded = 0
    with open(output_file, "wb") as f:
        for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
            f.write(chunk)
            downloaded += len(chunk)
            if total_bytes:
                pct = 100 * downloaded / total_bytes
                print(f"\r  Progress: {pct:.1f}% ({downloaded/1e9:.2f}/{total_bytes/1e9:.2f} GB)", end="")
    
    print(f"\n  Saved: {output_file} ({downloaded / 1e9:.2f} GB)")
    return output_file


# ============================================================
#  DOWNLOAD WEIGHTS — Method 2: Auto-discover from your account
# ============================================================

def list_and_download_best_checkpoint(output_dir="tinker_lora_download"):
    """List all your Tinker checkpoints, pick the best Nemotron one, and download it."""
    from tinker.types import ParsedCheckpointTinkerPath
    import tarfile
    
    service_client = tinker.ServiceClient()
    rest_client = service_client.create_rest_client()
    
    # List all checkpoints
    all_cps = []
    offset = 0
    while True:
        r = rest_client.list_user_checkpoints(limit=100, offset=offset).result()
        all_cps.extend(r.checkpoints)
        cur = r.cursor
        if cur is None or offset + cur.limit >= cur.total_count:
            break
        offset += cur.limit
    
    samplers = [c for c in all_cps if c.checkpoint_type == "sampler"]
    print(f"Found {len(samplers)} sampler checkpoints in your account:\n")
    
    for c in sorted(samplers, key=lambda x: x.time, reverse=True)[:10]:
        tid = ParsedCheckpointTinkerPath.from_tinker_path(c.tinker_path).training_run_id
        try:
            tr = rest_client.get_training_run(tid).result()
            bm = tr.base_model
        except Exception:
            bm = "?"
        print(f"  {c.time.isoformat()}  {bm}")
        print(f"    {c.tinker_path}\n")
    
    # Auto-pick: prefer "final" + "nemotron" sampler checkpoints
    candidates = []
    for c in samplers:
        parsed = ParsedCheckpointTinkerPath.from_tinker_path(c.tinker_path)
        tid = parsed.training_run_id
        try:
            tr = rest_client.get_training_run(tid).result()
        except Exception:
            continue
        if "nemotron" not in tr.base_model.lower():
            continue
        score = 0
        cid = c.checkpoint_id.lower()
        if "final" in cid:
            score += 3
        if "nemotron" in cid or "sft" in cid:
            score += 1
        candidates.append((score, c.time.timestamp(), c.tinker_path, c.checkpoint_id))
    
    if not candidates:
        print("No Nemotron sampler checkpoints found!")
        return None
    
    candidates.sort(key=lambda x: (-x[0], -x[1]))
    best_path = candidates[0][2]
    best_name = candidates[0][3]
    print(f"Auto-selected: {best_name}")
    print(f"  Path: {best_path}\n")
    
    # Download
    tar_file = download_checkpoint(service_client, best_path, f"{output_dir}.tar")
    
    # Extract
    import os
    os.makedirs(output_dir, exist_ok=True)
    with tarfile.open(tar_file, "r") as tar:
        tar.extractall(output_dir)
    print(f"  Extracted to: {output_dir}/")
    
    # Verify contents
    for fname in ["adapter_config.json", "adapter_model.safetensors"]:
        fpath = os.path.join(output_dir, fname)
        if os.path.exists(fpath):
            size = os.path.getsize(fpath)
            print(f"  {fname}: {size/1e6:.1f} MB")
        else:
            print(f"  WARNING: {fname} not found!")
    
    return output_dir


# ============================================================
#  TO DOWNLOAD, UNCOMMENT ONE OF THESE:
# ============================================================

# Method 1: You have the sampler_path from training
# download_checkpoint(service_client, sampler_path, "my_lora.tar")

# Method 2: Auto-discover from your account
# list_and_download_best_checkpoint("my_tinker_lora")

print("Download functions defined!")
print("Uncomment Method 1 or 2 above to download your weights.")

Download functions defined!
Uncomment Method 1 or 2 above to download your weights.


## Step 7: Quick Reference — The Full Pipeline in One Glance

```
┌─────────────────────────────────────────────────────────────┐
│  1. GET API KEY         tinker.thinkingmachines.ai          │
│  2. PREPARE DATASET     CSV/JSONL with <think> + \boxed{}   │
│  3. BUILD DATUMS        Manual builder (preserves reasoning)│
│  4. TRAIN ON TINKER     LoRA rank=32, 2 epochs, lr=2e-4    │
│  5. EVALUATE ON TINKER  Sampling API, no GPU needed         │
│  6. DOWNLOAD WEIGHTS    .tar → adapter_config + safetensors │
│  7. SUBMIT TO KAGGLE    zip at root level, done!            │
└─────────────────────────────────────────────────────────────┘
```

### Our Results

| Metric | Value |
|--------|-------|
| Dataset size | 10,000 rows |
| Training time | ~45 minutes on Tinker |
| Final train NLL | ~0.35 |
| Best eval NLL | ~0.28 |
| Hard 50 score | **47/50 (94%)** |
| LoRA size | ~1.5 GB (rank 32) |
| GPUs owned | **0** (all on Tinker) |

## Let's Beat 47/50 Together!

I'm sharing this because I believe the Kaggle community thrives when people share openly. My 47/50 is good, but there's room to improve:

- **Dataset quality** — Can we find even better reasoning chains? More diverse problem types?
- **Hyperparameter tuning** — Is there a better LR schedule? Should we try cosine decay?
- **Longer training** — Would 3 epochs help or hurt?
- **Data augmentation** — Multiple solutions per problem? Rejection sampling?
- **Ensemble tricks** — Alpha sweeps on the same weights?

**To experienced Tinker users:** If you see anything in my pipeline that could be improved, please drop a comment! We're all here to learn and push the boundaries of what's possible.

**To newcomers:** Don't be intimidated. This whole pipeline runs without a GPU. If you have a laptop and an internet connection, you can compete. That's the beauty of tools like Tinker.

**To Tinker:** Some extra credits grant will be appreciated for the free marketing lol. Just kidding but i deeply want to thank you guys for the 80k you offered to our comminity. Making the competition more fair for low budget competitors is a dream of many.

Let's keep sharing, keep improving, and keep building for the best of the community and the world.

Happy Kaggling! May your NLLs be low and your scores be high.